# Lesson 02 - Menjelajahi Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** adalah kerangka kerja terpadu untuk membangun agen AI. Ini menyediakan arsitektur yang bersih dan dapat disusun dengan empat blok bangunan inti:

- **Client** – menghubungkan ke endpoint model AI dan menangani komunikasi
- **Agent** – membungkus client dengan instruksi dan definisi alat
- **Tools** – memperluas kemampuan agen dengan fungsi kustom yang dapat dipanggil model
- **Session** – mempertahankan riwayat percakapan untuk interaksi multi-giliran

Dalam pelajaran ini, kita akan membangun **agen pemesanan perjalanan** yang memeriksa ketersediaan tujuan menggunakan konsep-konsep ini.


## Pengaturan


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Memahami Arsitektur Kerangka Agen

Kerangka Agen Microsoft mengikuti arsitektur berlapis:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` menghubungkan ke deployment Azure OpenAI. Ini menangani autentikasi, pemformatan permintaan, dan parsing respons.
2. **Agent** – Dibuat dari client melalui `provider.create_agent()`, agen menggabungkan akses model dengan instruksi (prompt sistem) dan alat.
3. **Tools** – Fungsi Python yang dihias dengan `@tool` yang dapat dipanggil agen untuk melakukan tindakan atau mengambil data.
4. **Session** – Objek `AgentSession` (dibuat melalui `agent.create_session()`) yang menyimpan riwayat percakapan, memungkinkan dialog multi-putar di mana agen mengingat konteks sebelumnya.

Mari kita bangun setiap lapisan satu per satu.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Menambahkan Alat dengan Dekorator @tool

Alat memungkinkan agen melakukan tindakan di luar menghasilkan teks. Dekorator `@tool` mengubah fungsi Python biasa menjadi sesuatu yang dapat dipanggil oleh agen.

Poin penting:
- Gunakan `Annotated[type, "description"]` agar model memahami setiap parameter.
- Docstring menjadi deskripsi alat yang dilihat model.
- `approval_mode="never_require"` berarti alat berjalan otomatis tanpa konfirmasi pengguna.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Membuat Agen dengan Alat

Sekarang kita menggabungkan client, instruksi, dan alat menjadi sebuah agen. `instructions` bertindak sebagai prompt sistem — mereka mendefinisikan persona dan perilaku agen.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Percakapan Multi-Turn dengan Sesi

Sebuah `AgentSession` (dibuat melalui `agent.create_session()`) melacak semua pesan dalam sebuah percakapan. Dengan melewatkan sesi yang sama ke setiap panggilan `agent.run()`, agen memiliki akses ke seluruh riwayat percakapan dan dapat merujuk kembali ke pesan-pesan sebelumnya.

Kami melewatkan `tools=[check_destination_availability]` sehingga agen dapat memanggil pemeriksa ketersediaan kami selama setiap giliran.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Ringkasan

Dalam pelajaran ini Anda mengeksplorasi empat pilar dari Microsoft Agent Framework:

| Konsep | Apa yang Anda Pelajari |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` terhubung ke Azure OpenAI dengan autentikasi berbasis kredensial |
| **Agent** | `provider.create_agent()` menggabungkan koneksi model dengan instruksi dan nama |
| **Tools** | Dekorator `@tool` mengekspos fungsi Python untuk dipanggil oleh agen |
| **Session** | `agent.create_session()` mempertahankan riwayat percakapan di beberapa giliran |

Blok bangunan ini saling menggabungkan untuk membuat agen yang dapat melakukan percakapan alami, memanggil fungsi eksternal, dan mempertahankan konteks — dasar untuk pola agen yang lebih maju di pelajaran berikutnya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya mencapai ketepatan, harap diperhatikan bahwa terjemahan otomatis dapat mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sahih. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau interpretasi yang salah yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
